# FastAPI - organizacja projektu

## APIRouter - czym są i po co?

**Router = podział endpointów na moduły**

### Bez routera (06.py):
```python
# main.py - wszystko w jednym pliku
app = FastAPI()

@app.get("/tasks")
def get_tasks(): ...

@app.post("/tasks")
def create_task(): ...

@app.get("/users")
def get_users(): ...

@app.post("/users")
def create_user(): ...
# ... 50 endpointów w jednym pliku!
```

**Problem:**
- Jeden plik ma 500+ linii
- Trudno znaleźć konkretny endpoint
- Nie da się pracować równolegle (git conflicts)
- Brak podziału logicznego (tasks vs users)

---

### Z routerem (app1):
```python
# routers/v1/tasks.py
router = APIRouter(prefix="/tasks", tags=["Tasks"])

@router.get("")           # GET /tasks
def get_tasks(): ...

@router.post("")          # POST /tasks
def create_task(): ...

# routers/v1/users.py
router = APIRouter(prefix="/users", tags=["Users"])

@router.get("")           # GET /users
def get_users(): ...

# main.py
from routers.v1 import tasks, users

app = FastAPI()
v1_router = APIRouter(prefix="/v1")
v1_router.include_router(tasks.router)
v1_router.include_router(users.router)
app.include_router(v1_router)
```

**Korzyści:**
- Podział logiczny - tasks w osobnym pliku, users w osobnym
- Łatwo znaleźć endpoint (wiem, że `/tasks` są w `routers/v1/tasks.py`)
- Praca równoległa - jeden dev w tasks.py, drugi w users.py
- Wersjonowanie API - łatwo dodać `/v2` (nowy folder `routers/v2/`)
- Czytelność - każdy plik ma 50-100 linii zamiast 500+

---

## Prefix i tags

**Prefix** - wspólny URL prefix dla wszystkich endpointów w routerze:
```python
router = APIRouter(prefix="/tasks")

@router.get("")           # GET /tasks (nie /!)
@router.get("/{id}")      # GET /tasks/{id}
@router.post("")          # POST /tasks
```

**Tags** - grupowanie w dokumentacji (Swagger):
```python
router = APIRouter(prefix="/tasks", tags=["Tasks"])
# W /docs wszystkie endpointy są pod sekcją "Tasks"
```

---

## Kiedy używać routera?

**TAK (zawsze w produkcji):**
- Projekt ma więcej niż 10 endpointów
- Endpointy dzielą się logicznie (tasks, users, products)
- Pracuje więcej niż 1 osoba
- Planujesz wersjonowanie API (/v1, /v2)

**NIE (tylko prototypy):**
- Prosty skrypt z 3 endpointami
- Jednorazowe demo/POC

**Zasada:** Jeśli projekt ma więcej niż 10 endpointów → użyj routera!

---

# Wartości domyślne parametrów w FastAPI

## TOP 5 - co wkładamy jako wartości domyślne parametrów

In [ ]:
from fastapi import Query, Path, Body, Depends, Header
from pydantic import BaseModel

# 1. Query - query params (?page=1&size=10)
@app.get("/tasks")
def get_tasks(page: int = Query(default=1, ge=1)):
    # GET /tasks?page=2
    ...

# 2. Path - path params (/tasks/{task_id})
@app.get("/tasks/{task_id}")
def get_task(task_id: int = Path(ge=1)):
    # GET /tasks/123
    ...

# 3. Body - JSON body (dla pojedynczych wartości)
@app.post("/tasks")
def create_task(name: str = Body(...)):
    # POST /tasks {"name": "Buy milk"}
    ...

# 4. Depends - uruchamia funkcję i wstrzykuje wynik
def get_db():
    session = Session(engine)
    yield session
    session.close()

@app.get("/tasks")
def get_tasks(db: Session = Depends(get_db)):
    # db = wynik z get_db()
    ...

# 5. Pydantic model - JSON body (dla struktur z wieloma polami)
class TaskCreate(BaseModel):
    name: str
    priority: int

@app.post("/tasks")
def create_task(task: TaskCreate):
    # POST /tasks {"name": "Buy milk", "priority": 5}
    ...

## Krótki opis każdego:

1. **Query** - `?param=value` - parametry z query string
2. **Path** - `/resource/{param}` - parametry z URL path
3. **Body** - JSON body - pojedyncze wartości (rzadko używane)
4. **Depends** - wstrzykiwanie zależności - uruchamia funkcję i wstrzykuje wynik
5. **Pydantic model** - JSON body - struktury z wieloma polami (najczęściej dla POST/PUT)

**Bonus:**
- **Header** - `Authorization: Bearer token` - HTTP headers
- **Cookie** - cookies
- **Form** - form data (multipart/form-data)
- **File/UploadFile** - file uploads

---

## Parametry walidacji - Path/Query/Field

**Path(), Query(), Body() pod spodem używają Pydantic Field()** - mają te same parametry:

| Parametr | Path/Query | Pydantic Field | Opis |
|----------|-----------|----------------|------|
| **ge** | ✅ | ✅ | greater or equal (>=) |
| **le** | ✅ | ✅ | less or equal (<=) |
| **gt** | ✅ | ✅ | greater than (>) |
| **lt** | ✅ | ✅ | less than (<) |
| **min_length** | ✅ | ✅ | minimalna długość str |
| **max_length** | ✅ | ✅ | maksymalna długość str |
| **pattern** | ✅ | ✅ | regex pattern |
| **default** | ✅ | ✅ | wartość domyślna |
| **description** | ✅ | ✅ | opis (w docs) |
| **example** | ✅ | ✅ | przykład (w docs) |
| **title** | ✅ | ✅ | tytuł (w docs) |
| **alias** | ✅ | ✅ | inna nazwa parametru |

In [ ]:
# Przykład - Path z walidacją
from fastapi import Path

@app.get("/tasks/{task_id}")
def get_task(
    task_id: int = Path(
        ge=1,
        le=1000,
        description="Unique task identifier",
        example=42
    )
):
    ...

In [ ]:
# To samo w Pydantic Field
from pydantic import BaseModel, Field

class Task(BaseModel):
    id: int = Field(
        ge=1,
        le=1000,
        description="Unique task identifier",
        example=42
    )

---

## Kiedy używać Path/Query, a kiedy Pydantic model?

### **Path/Query - dla pojedynczych wartości:**

```python
@app.get("/tasks/{task_id}")
def get_task(
    task_id: int = Path(ge=1),        # JEDNA wartość z URL
    page: int = Query(ge=1, default=1)  # JEDNA wartość z ?page=
):
    ...
```

**Dlaczego?**
- Prościej, czytelniej
- Nie trzeba tworzyć całej klasy dla jednej wartości

---

### **Pydantic model - dla struktur (wiele pól):**

```python
class TaskCreate(BaseModel):
    name: str = Field(min_length=3)
    description: str | None = None
    priority: int = Field(ge=1, le=5)
    tags: list[str] = []

@app.post("/tasks")
def create_task(task: TaskCreate):  # WIELE pól w JSON body
    ...
```

**Dlaczego?**
- JSON body często ma wiele pól
- Pydantic model grupuje je w jedną strukturę
- Łatwiej zarządzać walidacją

---

### **Czy można Pydantic dla Path/Query?**

**Teoretycznie TAK, ale to OVERKILL:**

```python
# ❌ DZIWNE - overkill dla jednej wartości
class TaskIdModel(BaseModel):
    task_id: int = Field(ge=1)

@app.get("/tasks/{task_id}")
def get_task(task_id: TaskIdModel):  # Niepotrzebnie skomplikowane!
    ...

# ✅ LEPIEJ - Path() dla pojedynczej wartości
@app.get("/tasks/{task_id}")
def get_task(task_id: int = Path(ge=1)):  # Czytelniej!
    ...
```

**Pydantic model dla path/query params = overkill!**
- Tworzy niepotrzebną klasę
- Mniej czytelne
- Normalne tylko dla JSON body (POST/PUT)

---

## Podsumowanie:

- **Path/Query** = pojedyncze wartości (uproszczenie)
- **Pydantic model** = struktury z wieloma polami (JSON body)
- **Path/Query używają Pydantic Field pod spodem** (te same parametry: ge, le, min_length, etc.)
- **Nie używaj Pydantic modelu dla path/query** - to niepotrzebne komplikowanie!

---

## Annotated - metadata dla typów

### Ogólny Python (ignoruje metadata):

In [ ]:
from typing import Annotated, get_args

# Annotated = typ + metadata (dowolne dodatkowe info)
x: Annotated[int, "to jest int większy niż 0"] = 5
#              ^^^  ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
#              typ        metadata (opis)

# Python runtime:
print(x)  # 5
print(type(x))  # <class 'int'>

# Python IGNORUJE metadata w runtime!
def foo(x: Annotated[int, "większe niż 0"]):
    print(x)

foo(-5)  # Działa! Python nie sprawdza "większe niż 0"

# Metadata są w __annotations__:
print(foo.__annotations__)
# {'x': typing.Annotated[int, 'większe niż 0']}

# Można wyciągnąć metadata:
annotation = foo.__annotations__['x']
base_type, *metadata = get_args(annotation)
print(base_type)  # <class 'int'>
print(metadata)   # ['większe niż 0']

**Python:**
- Wkłada `Annotated` do `__annotations__`
- Runtime **kompletnie ignoruje** metadata
- To tylko "pudełko na metadata" dla zewnętrznych narzędzi (FastAPI, Pydantic, mypy)

---

### FastAPI (czyta i używa metadata):

In [ ]:
from typing import Annotated
from fastapi import Path, Query, Depends

# FastAPI czyta metadata z Annotated i używa ich!

# BEZ Annotated - powtarzamy walidację
@app.get("/tasks/{task_id}")
def get_task(task_id: int = Path(ge=1)):
    ...

@app.put("/tasks/{task_id}")
def update_task(task_id: int = Path(ge=1)):  # Powtórzenie!
    ...

@app.delete("/tasks/{task_id}")
def delete_task(task_id: int = Path(ge=1)):  # Powtórzenie!
    ...

# Z Annotated - definiujemy raz, używamy wszędzie (DRY!)
PositiveInt = Annotated[int, Path(ge=1)]
#                        ^^^  ^^^^^^^^^^^
#                        typ   metadata (FastAPI używa Path(ge=1)!)

@app.get("/tasks/{task_id}")
def get_task(task_id: PositiveInt):  # FastAPI czyta Path(ge=1) z metadata
    ...

@app.put("/tasks/{task_id}")
def update_task(task_id: PositiveInt):  # Reużywamy!
    ...

@app.delete("/tasks/{task_id}")
def delete_task(task_id: PositiveInt):  # Reużywamy!
    ...

**Te dwa są RÓWNOWAŻNE dla FastAPI:**

```python
# Bez Annotated
task_id: int = Path(ge=1)

# Z Annotated
PositiveInt = Annotated[int, Path(ge=1)]
task_id: PositiveInt
```

FastAPI:
1. Widzi `Annotated[int, Path(ge=1)]`
2. Wyciąga `Path(ge=1)` z metadata (używa `get_args()`)
3. Używa jako walidację (tak samo jak `= Path(ge=1)`)

---

### Przykłady praktyczne:

In [ ]:
# dependencies.py - centralizacja
from typing import Annotated
from fastapi import Path, Query, Depends
from sqlalchemy.orm import Session
from database import get_db

# Path params
PositiveInt = Annotated[int, Path(ge=1)]

# Query params
PageNumber = Annotated[int, Query(ge=1, default=1)]
PageSize = Annotated[int, Query(ge=1, le=100, default=10)]

# Depends
DbSession = Annotated[Session, Depends(get_db)]

# Użycie w routerze:
@app.get("/tasks/{task_id}")
def get_task(
    task_id: PositiveInt,  # = Path(ge=1)
    db: DbSession          # = Depends(get_db)
):
    ...

@app.get("/tasks")
def get_tasks(
    page: PageNumber,      # = Query(ge=1, default=1)
    size: PageSize,        # = Query(ge=1, le=100, default=10)
    db: DbSession          # = Depends(get_db)
):
    ...

---

## Podsumowanie Annotated:

**Ogólny Python:**
- `Annotated[typ, metadata]` to "pudełko" na typ + dodatkowe info
- Python runtime **ignoruje** metadata (nie sprawdza, nie waliduje)
- Metadata są w `__annotations__` (dla zewnętrznych narzędzi)

**FastAPI:**
- **Czyta** metadata z `Annotated`
- Używa ich jak wartości domyślne (`Path(ge=1)`, `Depends(get_db)`)
- **Korzyści:**
  - DRY - definiujesz raz, używasz wszędzie
  - Centralizacja - zmiana w jednym miejscu (`dependencies.py`)
  - Czytelność - `task_id: PositiveInt` zamiast `task_id: int = Path(ge=1)`

**Równoważne zapisy:**

```python
# Bez Annotated
x: int = Path(ge=1)

# Z Annotated
PositiveInt = Annotated[int, Path(ge=1)]
x: PositiveInt
```